# Milestone 2: Data Quality dan Profiling

Lanjutkan dari hasil `01_data_profiling.ipynb`. Notebook ini menerapkan profiling DuckDB `SUMMARIZE` dan
transformasi Polars untuk menangani missing values, outlier, duplikat, serta validasi data dasar
pada dataset revisi Wikipedia Bahasa Indonesia (`idwiki_revisions.parquet`, 27.771.250 baris × 15 kolom).

| Item | Nilai |
|---|---|
| Milestone | **Milestone 2** (target check-in Pertemuan 5) |
| Deliverable | DuckDB `SUMMARIZE`, transformasi Polars, penanganan missing values & outlier, validasi data dasar |
| Rubrik | Data handling Polars/DuckDB (25%) + Cleaning & data quality (15%) |

## Prinsip penanganan (penting)

Berdasarkan profiling Milestone 1, **mayoritas null pada dataset ini bersifat struktural, bukan kesalahan data**:
`parent_rev_id` null = revisi pertama halaman; `comment` null = suntingan tanpa keterangan;
`contributor_id` null = penyunting anonim (IP). Karena itu penanganan utama bukan imputasi nilai,
melainkan **kolom flag** yang mempertahankan makna. Hanya `contributor_name` (null 0,02%) yang diisi
label eksplisit.

Polars dipakai dalam mode **lazy/streaming**; DuckDB dibatasi `memory_limit` agar aman di container ~3,7 GB.


In [1]:
from pathlib import Path
import time

import duckdb
import polars as pl

DATA_PATH = Path('data/processed/idwiki_revisions.parquet')
if not DATA_PATH.exists():  # fallback bila kernel berjalan dari folder notebooks/
    DATA_PATH = Path('..') / DATA_PATH
assert DATA_PATH.exists(), f'File tidak ditemukan: {DATA_PATH.resolve()}'

CLEAN_PATH = Path('data/processed/idwiki_revisions_clean.parquet')
if not CLEAN_PATH.exists():
    CLEAN_PATH = Path('..') / CLEAN_PATH

lf = pl.scan_parquet(DATA_PATH)  # lazy: belum baca data ke memori
con = duckdb.connect()           # in-memory DuckDB
con.execute("SET memory_limit='800MB'")   # batasi DuckDB agar aman di RAM container ~4 GB
con.execute("SET temp_directory='/tmp'")    # spill perhitungan besar ke disk, bukan ke repo
con.execute(f"CREATE OR REPLACE VIEW revisi AS SELECT * FROM read_parquet('{DATA_PATH.as_posix()}')")

print('DATA_PATH  :', DATA_PATH)
print('ukuran     :', f'{DATA_PATH.stat().st_size / 1e9:.2f} GB')
print('polars     :', pl.__version__)
print('duckdb     :', duckdb.__version__)

DATA_PATH  : ../data/processed/idwiki_revisions.parquet
ukuran     : 1.96 GB
polars     : 2.0.0
duckdb     : 1.5.6


## 1. Profiling dengan DuckDB `SUMMARIZE`

`SUMMARIZE` memberikan profil lengkap tiap kolom (tipe, null, jumlah unik, nilai minimum/maksimum, contoh)
dalam **satu scan** — deliverable wajib Milestone 2.


In [2]:
t0 = time.perf_counter()
profil = con.execute('SUMMARIZE SELECT * FROM revisi').df()
print(f'SUMMARIZE selesai dalam {time.perf_counter() - t0:.1f} detik')
profil

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

SUMMARIZE selesai dalam 70.5 detik


,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,page_id,BIGINT,1,4809113,3984056,1581508.4753069452,1386463.3607054928,296132,1244372,2720204,27771250,0.00
1,page_title,VARCHAR,!,🫏,5206998,NaN,NaN,NaN,NaN,NaN,27771250,0.00
2,page_namespace,SMALLINT,0,2600,31,2.8731309177656748,47.36190317193622,0,0,1,27771250,0.00
3,is_redirect,BOOLEAN,false,true,2,NaN,NaN,NaN,NaN,NaN,27771250,0.00
4,rev_id,BIGINT,1,29684531,37862751,14946897.800199343,8588505.21756407,7494334,14994457,22451454,27771250,0.00
5,parent_rev_id,BIGINT,1,29684494,27192251,14344437.404395927,8435079.923323363,6961260,14366114,21381099,27771250,15.32
6,rev_timestamp,TIMESTAMP,2003-01-24 16:06:03,2026-09-01 12:57:27,30313789,2018-04-11 04:49:07.71821,NaN,2013-12-20 20:36:14.994947,2019-04-06 14:31:18.108242,2022-12-25 20:02:29.026044,27771250,0.00
7,contributor_name,VARCHAR,!!ndo123456789,𥐣𥐣𥐣,882668,NaN,NaN,NaN,NaN,NaN,27771250,0.02
8,contributor_id,BIGINT,0,1743392,360419,492361.5243114042,479693.0991099087,140326,232193,822106,27771250,7.92
9,is_anonymous,BOOLEAN,false,true,2,NaN,NaN,NaN,NaN,NaN,27771250,0.00


In [3]:
# Diskusi ringkas hasil SUMMARIZE (dihitung ulang dari tabel profil)
import pandas as pd

# DuckDB >=1.4 tidak lagi mengekspos kolom `null_count`; derivasi dari null_percentage
n_rows_summ = int(con.execute('SELECT COUNT(*) FROM revisi').fetchone()[0])
ringkas = profil[['column_name', 'column_type', 'null_percentage', 'approx_unique']].copy()
ringkas['null_count'] = (ringkas['null_percentage'] / 100.0 * n_rows_summ).round().astype('int64')
ringkas['null_percentage'] = ringkas['null_percentage'].round(3)
ringkas = ringkas[['column_name', 'column_type', 'null_count', 'null_percentage', 'approx_unique']]
ringkas = ringkas.sort_values('null_count', ascending=False).reset_index(drop=True)
ringkas

,column_name,column_type,null_count,null_percentage,approx_unique
0,comment,VARCHAR,5251543,18.91,6698733
1,parent_rev_id,BIGINT,4254556,15.32,27192251
2,contributor_id,BIGINT,2199483,7.92,360419
3,rev_sha1,VARCHAR,8331,0.03,27109994
4,contributor_name,VARCHAR,5554,0.02,882668
5,page_title,VARCHAR,0,0.00,5206998
6,page_id,BIGINT,0,0.00,3984056
7,rev_timestamp,TIMESTAMP,0,0.00,30313789
8,rev_id,BIGINT,0,0.00,37862751
9,is_redirect,BOOLEAN,0,0.00,2


## 2. Deteksi Missing Values

Lihat jumlah dan persentase null per kolom sebelum memutuskan strategi penanganan.
Pada dataset ini null bersifat **struktural** (bukan kesalahan) — dibuktikan lewat cek konsistensi
`parent_rev_id` terhadap jumlah halaman unik di bawah.


In [4]:
laporan = con.execute('''
    SELECT
        COUNT(*) AS total_baris,

        -- Unik / duplikasi
        COUNT(DISTINCT rev_id) AS rev_id_unik,
        COUNT(*) - COUNT(DISTINCT rev_id) AS rev_id_duplikat,

        -- Missing values per kolom
        SUM(CASE WHEN page_id IS NULL THEN 1 ELSE 0 END) AS null_page_id,
        SUM(CASE WHEN page_title IS NULL THEN 1 ELSE 0 END) AS null_page_title,
        SUM(CASE WHEN rev_id IS NULL THEN 1 ELSE 0 END) AS null_rev_id,
        SUM(CASE WHEN rev_timestamp IS NULL THEN 1 ELSE 0 END) AS null_timestamp,
        SUM(CASE WHEN parent_rev_id IS NULL THEN 1 ELSE 0 END) AS null_parent,
        SUM(CASE WHEN comment IS NULL THEN 1 ELSE 0 END) AS null_comment,
        SUM(CASE WHEN contributor_id IS NULL THEN 1 ELSE 0 END) AS null_contributor_id,
        SUM(CASE WHEN contributor_name IS NULL THEN 1 ELSE 0 END) AS null_contributor_name,
        SUM(CASE WHEN rev_sha1 IS NULL THEN 1 ELSE 0 END) AS null_rev_sha1,

        -- Nilai tidak valid (harusnya mustahil)
        SUM(CASE WHEN text_bytes < 0 THEN 1 ELSE 0 END) AS text_bytes_negatif,
        SUM(CASE WHEN page_namespace < 0 THEN 1 ELSE 0 END) AS namespace_negatif,
        SUM(CASE WHEN rev_timestamp < DATE '2001-01-01' THEN 1 ELSE 0 END) AS timestamp_terlalu_tua,
        SUM(CASE WHEN rev_timestamp > DATE '2027-01-01' THEN 1 ELSE 0 END) AS timestamp_masa_depan,

        -- Flag boolean yang tidak konsisten
        SUM(CASE WHEN is_anonymous AND contributor_id IS NOT NULL THEN 1 ELSE 0 END) AS anonim_tapi_punya_id,
        SUM(CASE WHEN (NOT is_anonymous) AND contributor_id IS NULL AND contributor_name IS NOT NULL
                      AND contributor_name ~ '^[0-9]{1,3}(\\.[0-9]{1,3}){3}$'
                 THEN 1 ELSE 0 END) AS nama_ip_tapi_tidak_anonim
    FROM revisi
''').df()
laporan.T

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,0
total_baris,27771250.0
rev_id_unik,27771250.0
rev_id_duplikat,0.0
null_page_id,0.0
null_page_title,0.0
null_rev_id,0.0
null_timestamp,0.0
null_parent,4255798.0
null_comment,5252300.0
null_contributor_id,2199136.0


In [5]:
# Buktikan: jumlah halaman yang minimal punya satu revisi dengan parent null
# ~= jumlah halaman unik (revisi pertama tiap halaman tidak punya parent)
bukti = con.execute('''
    SELECT
        COUNT(DISTINCT page_id) AS halaman_unik,
        COUNT(DISTINCT CASE WHEN parent_rev_id IS NULL THEN page_id END) AS halaman_dengan_revisi_tanpa_parent,
        COUNT(*) AS revisi_tanpa_parent
    FROM revisi
''').df()
bukti

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,halaman_unik,halaman_dengan_revisi_tanpa_parent,revisi_tanpa_parent
0,4253661,4253591,27771250


## 3. Penanganan Missing Values

Karena null bersifat struktural, kita **tidak melakukan imputasi buta**. Sebagai gantinya:

- Null `parent_rev_id`, `comment`, `contributor_id` diwakili **kolom flag**
  (`is_first_revision`, `has_comment`, `is_registered_user`) agar makna tetap jelas.
- `contributor_name` null (0,02%) diisi label eksplisit `"unknown"`.

Preview kolom derivasi (lazy) di bawah; implementasi penuh ada di pipeline §7.


In [6]:
# Preview kolom derivasi (lazy — hanya dibaca kolom yang dibutuhkan)
preview_derivasi = (
    lf.select([
        'rev_id', 'parent_rev_id', 'comment', 'contributor_id', 'contributor_name',
        pl.col('parent_rev_id').is_null().alias('is_first_revision'),
        pl.col('comment').is_not_null().alias('has_comment'),
        pl.col('contributor_id').is_not_null().alias('is_registered_user'),
    ])
    .head(8)
    .collect()
)
preview_derivasi

rev_id,parent_rev_id,comment,contributor_id,contributor_name,is_first_revision,has_comment,is_registered_user
i64,i64,str,i64,str,bool,bool,bool
1,null,"""*""",0,"""proxy8.ntu.edu.sg""",true,true,true
2,1,"""*""",0,"""proxy8.ntu.edu.sg""",false,true,true
3,2,"""*""",0,"""155.69.5.xxx""",false,true,true
4,3,"""*""",0,"""155.69.5.xxx""",false,true,true
5,4,"""*""",0,"""155.69.5.xxx""",false,true,true
6,5,"""*Tambahkan 'link' ke definisi …",756234,"""Vyasa""",false,true,true
7,6,"""*Tambahkan gambar DNA""",756234,"""Vyasa""",false,true,true
86,7,"""*Perbaiki garis bawah (ubah ""<…",756234,"""Vyasa""",false,true,true


## 4. Deteksi dan Penanganan Outlier

Distribusi `text_bytes` sangat miring ke kanan (median ±2,5 KB vs maks ±2,1 MB). Outlier ditandai
dengan batas **IQR** dan **persentil p1/p99** (winsorization), ditambah domain knowledge: ukuran
artikel Wikipedia hingga beberapa MB masih wajar, sehingga **baris tidak dihapus** — hanya ditandai
dan di-clamp untuk analisis statistik lanjutan.


In [7]:
stats = con.execute('''
    SELECT
        quantile_cont(text_bytes, 0.01) AS p1,
        quantile_cont(text_bytes, 0.25) AS q1,
        quantile_cont(text_bytes, 0.50) AS median,
        quantile_cont(text_bytes, 0.75) AS q3,
        quantile_cont(text_bytes, 0.99) AS p99,
        AVG(text_bytes) AS mean,
        MAX(text_bytes) AS max,
        SUM(CASE WHEN text_bytes = 0 THEN 1 ELSE 0 END) AS n_zero,
        SUM(CASE WHEN text_bytes > 100000 THEN 1 ELSE 0 END) AS n_gt_100kb,
        SUM(CASE WHEN text_bytes > 1000000 THEN 1 ELSE 0 END) AS n_gt_1mb
    FROM revisi
''').df()
iqr = float(stats['q3'].iloc[0]) - float(stats['q1'].iloc[0])
batas_iqr_bawah = float(stats['q1'].iloc[0]) - 1.5 * iqr
batas_iqr_atas = float(stats['q3'].iloc[0]) + 1.5 * iqr
print(f'IQR = {iqr:,.0f} | Batas IQR: [{batas_iqr_bawah:,.0f}, {batas_iqr_atas:,.0f}]')
print(f'Batas winsor (p1-p99): [{float(stats["p1"].iloc[0]):,.0f}, {float(stats["p99"].iloc[0]):,.0f}]')
print('Domain: 0 B = revisi tanpa meta ukuran (deletions/redirect) | >1 MB = suntingan artikel sangat besar (wajar tapi jarang)')
stats

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

IQR = 5,790 | Batas IQR: [-7,705, 15,455]
Batas winsor (p1-p99): [20, 115,873]
Domain: 0 B = revisi tanpa meta ukuran (deletions/redirect) | >1 MB = suntingan artikel sangat besar (wajar tapi jarang)


,p1,q1,median,q3,p99,mean,max,n_zero,n_gt_100kb,n_gt_1mb
0,20.0,980.0,2516.0,6770.0,115873.0,9251.012579,2096646,53922.0,361137.0,1877.0


## 5. Deduplikasi

`rev_id` adalah kunci alami (ID revisi MediaWiki global). Bila unik, tidak ada duplikat baris yang berarti.
`unique()` tetap diterapkan pada pipeline sebagai *guard rail* dokumentatif.


In [8]:
n_rows = lf.select(pl.len()).collect().item()
n_rev_unik = lf.select(pl.col('rev_id').n_unique()).collect().item()
print(f'Baris          : {n_rows:,}')
print(f'rev_id unik    : {n_rev_unik:,}')
print(f'Duplikat kunci : {n_rows - n_rev_unik:,}')
assert n_rows == n_rev_unik, 'rev_id tidak unik — perlu deduplikasi serius di pipeline!'

Baris          : 27,771,250
rev_id unik    : 27,771,250
Duplikat kunci : 0


## 6. Validasi Data Dasar

Tulis aturan validasi untuk memastikan data bersih memenuhi batas yang masuk akal.
Pada notebook ini validasi dijalankan dua lapis:

1. **Great Expectations** — *suite* aturan deklaratif pada sampel deterministik ~300 ribu baris (hemat memori).
2. **DuckDB aturan global** — dihitung penuh pada seluruh 27,7 juta baris (unik `rev_id`, null kunci, rentang nilai).

Ditambah pemeriksaan konsistensi (referensial `parent_rev_id`, distribusi namespace).


In [9]:
# Referensial: parent_rev_id (yang terisi) harus merujuk ke rev_id yang ada
t0 = time.perf_counter()
parent_check = con.execute('''
    WITH parents AS (
        SELECT DISTINCT parent_rev_id AS rev_id FROM revisi WHERE parent_rev_id IS NOT NULL
    )
    SELECT
        (SELECT COUNT(*) FROM parents) AS parent_unik_terisi,
        (SELECT COUNT(*) FROM parents p ANTI JOIN (SELECT rev_id FROM revisi) r ON p.rev_id = r.rev_id) AS parent_menggantung
''').df()
print(f'Pemeriksaan referensial selesai dalam {time.perf_counter() - t0:.1f} detik')
parent_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pemeriksaan referensial selesai dalam 14.7 detik


,parent_unik_terisi,parent_menggantung
0,23515014,52


In [10]:
# Distribusi namespace dan nilai di luar namespace standar MediaWiki (0-15 + turunan)
ns_dist = con.execute('''
    SELECT
        page_namespace,
        COUNT(*) AS jumlah_revisi,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 3) AS persen
    FROM revisi
    GROUP BY page_namespace
    ORDER BY jumlah_revisi DESC
''').df()
print('Jumlah nilai page_namespace unik:', ns_dist.shape[0])
print('Revisi dengan namespace > 15 (di luar standar):',
      int(ns_dist.loc[ns_dist.page_namespace > 15, 'jumlah_revisi'].sum()))
ns_dist.head(15)

Jumlah nilai page_namespace unik: 28
Revisi dengan namespace > 15 (di luar standar): 76163


,page_namespace,jumlah_revisi,persen
0,0,20358320,73.307
1,3,2181462,7.855
2,1,2054374,7.397
3,14,1061123,3.821
4,10,650851,2.344
5,4,502038,1.808
6,2,501678,1.806
7,6,222346,0.801
8,15,74132,0.267
9,11,45688,0.165


### Validasi otomatis Great Expectations

Great Expectations (GX) menjalankan *suite* aturan deklaratif terhadap data. Karena dataset 27,7 juta baris,
ekspektasi **per-baris** dijalankan pada **sampel deterministik ~300 ribu baris** (filter hash tetap pada `rev_id`,
hanya kolom yang diuji — hemat memori), sedangkan aturan yang **harus dihitung penuh** (unik `rev_id`, jumlah null
global, rentang global) diverifikasi terpisah lewat DuckDB pada sub-bagian berikutnya. Kombinasi ini menjaga
validasi tetap komprehensif tanpa risiko OOM.


In [11]:
try:
    import great_expectations as gx
    print('Great Expectations:', gx.__version__)
    GX_AVAILABLE = True
except ImportError:
    print('Great Expectations tidak tersedia — fallback validasi DuckDB pada Bagian 8.')
    GX_AVAILABLE = False

Great Expectations: 1.24.0


In [12]:
# Sampel deterministik ~300 ribu baris — hanya kolom yang diuji (hemat memori)
# Polars 2.x LazyFrame tidak punya .sample; pakai filter hash deterministik pada rev_id.
# Catatan: hash() wajib ? modulo pada nilai mentah rev_id bias ke revisi awal (rev_id < modulus).
GX_SAMPLE_N = 300_000
kolom_gx = [
    'rev_id', 'rev_timestamp', 'page_namespace', 'text_bytes',
    'is_minor', 'is_anonymous', 'contributor_name', 'parent_rev_id',
]
t0 = time.perf_counter()
n_total_lf = lf.select(pl.len()).collect().item()
fraction = min(GX_SAMPLE_N / n_total_lf, 1.0)
sample = (
    lf.filter((pl.col('rev_id').hash() % 1_000_000_007) < int(fraction * 1_000_000_007))  # hash(): deterministik tapi teracak merata
    .select(kolom_gx)
    .collect()
)
df_sample = sample.to_pandas()
del sample            # bebaskan buffer Polars sebelum proses GX
import gc; gc.collect()
print(f'Sampel {len(df_sample):,} baris x {df_sample.shape[1]} kolom (fraction={fraction:.4f}) dalam {time.perf_counter() - t0:.1f} detik')
df_sample.head(3)

Sampel 299,310 baris x 8 kolom (fraction=0.0108) dalam 3.8 detik


,rev_id,rev_timestamp,page_namespace,text_bytes,is_minor,is_anonymous,contributor_name,parent_rev_id
0,1162644,2008-01-14 09:55:39,0,11734,True,False,Alexbot,1162379.0
1,5055335,2011-12-23 10:46:57,0,14668,True,False,WikitanvirBot,5032946.0
2,11403917,2016-03-13 11:29:18,0,36946,True,False,Wagino Bot,8852534.0


In [13]:
hasil_validasi = None

if GX_AVAILABLE:
    ctx = gx.get_context(mode='ephemeral')

    ds = ctx.data_sources.add_pandas(name='idwiki_bersih')
    da = ds.add_dataframe_asset(name='revisi_sample')
    batch_def = da.add_batch_definition_whole_dataframe('seluruh_sampel')

    suite = ctx.suites.add(gx.ExpectationSuite(name='aturan_revisi_idwiki'))

    # 1. Teks tidak negatif dan tidak absurd (>5 MB dianggap tak wajar untuk 1 revisi)
    suite.add_expectation(
        gx.expectations.ExpectColumnValuesToBeBetween(
            column='text_bytes', min_value=0, max_value=5_000_000)
    )
    # 2. Namespace tidak negatif dan tidak melampaui batas wajar
    suite.add_expectation(
        gx.expectations.ExpectColumnValuesToBeBetween(
            column='page_namespace', min_value=0, max_value=3_000)
    )
    # 3. Kunci revisi wajib terisi
    suite.add_expectation(
        gx.expectations.ExpectColumnValuesToNotBeNull(column='rev_id')
    )
    suite.add_expectation(
        gx.expectations.ExpectColumnValuesToNotBeNull(column='rev_timestamp')
    )
    # 4. rev_id unik pada sampel (indikasi duplikasi masif bila gagal)
    suite.add_expectation(
        gx.expectations.ExpectColumnValuesToBeUnique(column='rev_id')
    )
    # 5. Flag boolean hanya berisi TRUE/FALSE
    suite.add_expectation(
        gx.expectations.ExpectColumnValuesToBeInSet(
            column='is_minor', value_set=[True, False])
    )
    suite.add_expectation(
        gx.expectations.ExpectColumnValuesToBeInSet(
            column='is_anonymous', value_set=[True, False])
    )

    vd = ctx.validation_definitions.add(
        gx.ValidationDefinition(name='validasi_utama', data=batch_def, suite=suite)
    )
    t0 = time.perf_counter()
    hasil_validasi = vd.run(batch_parameters={'dataframe': df_sample})
    print(f'Validasi GX selesai dalam {time.perf_counter() - t0:.1f} detik')
    print('Status keseluruhan:', 'LULUS' if hasil_validasi.success else 'GAGAL')
else:
    print('Melewati GX — lihat Bagian 8.')

Calculating Metrics:   0%|          | 0/46 [00:00<?, ?it/s]

Validasi GX selesai dalam 0.6 detik
Status keseluruhan: LULUS


In [14]:
if hasil_validasi is not None:
    baris = []
    for ev in hasil_validasi.results:
        cfg = ev.expectation_config
        meta = getattr(ev, 'result', {}) or {}
        baris.append({
            'aturan': cfg.type,
            'kolom': (cfg.kwargs or {}).get('column', ''),
            'lulus': bool(ev.success),
            'jumlah_gagal': meta.get('unexpected_count', None),
        })
    df_gx = pl.DataFrame(baris)
    print(f"{int(df_gx['lulus'].sum())}/{len(df_gx)} aturan lulus")
    df_gx
else:
    print('Tidak ada hasil GX.')

7/7 aturan lulus


### Validasi global DuckDB (seluruh baris)

Aturan yang tidak boleh hanya diuji pada sampel — dihitung penuh dan dilaporkan lulus/gagal.


In [15]:
aturan_global = con.execute('''
    SELECT 'rev_id unik (tanpa duplikat)' AS aturan,
           COUNT(*) = COUNT(DISTINCT rev_id) AS lulus,
           COUNT(*) - COUNT(DISTINCT rev_id) AS pelanggaran
    FROM revisi
    UNION ALL
    SELECT 'rev_id tidak boleh null', COUNT(*) = COUNT(rev_id), COUNT(*) - COUNT(rev_id) FROM revisi
    UNION ALL
    SELECT 'rev_timestamp tidak boleh null', COUNT(*) = COUNT(rev_timestamp), COUNT(*) - COUNT(rev_timestamp) FROM revisi
    UNION ALL
    SELECT 'text_bytes >= 0', SUM(text_bytes >= 0), SUM(text_bytes < 0) FROM revisi
    UNION ALL
    SELECT 'page_namespace >= 0', SUM(page_namespace >= 0), SUM(page_namespace < 0) FROM revisi
    UNION ALL
    SELECT 'timestamp 2001-2027 (rentang wajar)',
           SUM(rev_timestamp BETWEEN DATE '2001-01-01' AND DATE '2027-01-01'),
           SUM(NOT (rev_timestamp BETWEEN DATE '2001-01-01' AND DATE '2027-01-01')) FROM revisi
    UNION ALL
    SELECT 'tidak ada anonim yang punya contributor_id',
           SUM(CASE WHEN is_anonymous AND contributor_id IS NOT NULL THEN 0 ELSE 1 END),
           SUM(CASE WHEN is_anonymous AND contributor_id IS NOT NULL THEN 1 ELSE 0 END) FROM revisi
''').df()
print('Semua aturan global lulus:', bool(aturan_global['lulus'].all()))
aturan_global

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Semua aturan global lulus: True


,aturan,lulus,pelanggaran
0,rev_id unik (tanpa duplikat),1.0,0.0
1,rev_id tidak boleh null,1.0,0.0
2,rev_timestamp tidak boleh null,1.0,0.0
3,text_bytes >= 0,27771250.0,0.0
4,page_namespace >= 0,27771250.0,0.0
5,timestamp 2001-2027 (rentang wajar),27771250.0,0.0
6,tidak ada anonim yang punya contributor_id,27771250.0,0.0


## 7. Simpan Data Bersih

Satu fungsi pipeline menerapkan seluruh keputusan penanganan secara **reproducible**, lalu disimpan
secara streaming:

1. Guardrail deduplikasi `rev_id`
2. Kolom derivasi struktural (`is_first_revision`, `has_comment`, `is_registered_user`)
3. Isi `contributor_name` null → `"unknown"`
4. Batas winsor `text_bytes` (p1/p99) + flag outlier
5. Tidak ada baris yang dihapus — seluruh 27,7 juta revisi tetap utuh

Simpan ke `data/processed/` agar dipakai Milestone 3 tanpa mengulang proses cleaning.


In [16]:
# Hitung batas winsor global (p1, p99) sekali
batas = lf.select(
    pl.col('text_bytes').quantile(0.01).alias('p1'),
    pl.col('text_bytes').quantile(0.99).alias('p99'),
).collect()
P1 = float(batas['p1'][0])
P99 = float(batas['p99'][0])
print(f'Batas winsor text_bytes: p1={P1:,.0f}  p99={P99:,.0f}')


def pipeline_data_cleaning(lf_input: pl.LazyFrame, p1: float = P1, p99: float = P99, dedup: bool = True) -> pl.LazyFrame:
    """Pipeline cleaning Milestone 2 untuk dataset revisi idwiki.

    Keputusan penanganan:
    - Null struktural (parent_rev_id, comment, contributor_id) TIDAK diisi;
      diwakili kolom flag agar makna tetap jelas bagi analisis lanjutan.
    - contributor_name null (0,02%) diisi label 'unknown'.
    - Outlier text_bytes ditandai (IQR domain) dan di-winsor (p1..p99)
      tanpa menghapus baris.
    """
    base = lf_input
    if dedup:
        # 1. Guardrail: pertahankan baris pertama per rev_id
        #    (rev_id terbukti unik; tanpa maintain_order agar mendukung sink streaming)
        base = base.unique(subset=['rev_id'], keep='first')
    return (
        base
        # 2. Kolom derivasi + pembersihan teks
        .with_columns([
            pl.col('parent_rev_id').is_null().alias('is_first_revision'),
            pl.col('comment').is_not_null().alias('has_comment'),
            pl.col('contributor_id').is_not_null().alias('is_registered_user'),
            pl.col('contributor_name').fill_null('unknown').alias('contributor_name'),
        ])
        # 3. Penanganan outlier text_bytes
        .with_columns([
            ((pl.col('text_bytes') < p1) | (pl.col('text_bytes') > p99)).alias('is_outlier_size'),
            pl.col('text_bytes').clip(lower_bound=p1, upper_bound=p99).alias('text_bytes_winsorized'),
        ])
    )


lf_bersih = pipeline_data_cleaning(lf)
print('Pipeline siap dieksekusi (lazy).')

Batas winsor text_bytes: p1=20  p99=115,873
Pipeline siap dieksekusi (lazy).


In [17]:
# Eksekusi pipeline: tulis STREAMING dengan sink_parquet -
# 27,7 juta baris TIDAK pernah dimuat sekaligus ke RAM (aman untuk container ~3,7 GB).
#
# Tahap pelepasan memori sebelum sink (wajib di RAM kecil):
# - DuckDB ditutup -> buffer internal (s.d. 800 MB) dilepas.
# - Objek berat Bagian 1-7 (sampel pandas, hasil GX) dibuang lalu garbage-collected.
con.close()
for _nama in ('df_sample', 'hasil_validasi', 'df_gx'):
    globals().pop(_nama, None)
import gc; gc.collect()

t0 = time.perf_counter()
CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
# dedup=False: uniknya rev_id sudah diverifikasi dua kali (assert Bagian 5 + aturan
# global DuckDB Bagian 8), sehingga tahap unique() dilepas agar eksekusi tetap
# murni streaming dan hemat memori.
lf_bersih = pipeline_data_cleaning(lf, dedup=False)
lf_bersih.sink_parquet(CLEAN_PATH, compression='zstd')
waktu = time.perf_counter() - t0
print(f'Pipeline dieksekusi (streaming sink_parquet) dalam {waktu / 60:.2f} menit')

# Statistik hasil dihitung ulang dari lazy scan file hasil
lf_clean = pl.scan_parquet(CLEAN_PATH)
n_baris_bersih = lf_clean.select(pl.len()).collect().item()
n_kolom_bersih = len(lf_clean.collect_schema())
print(f'Hasil: {n_baris_bersih:,} baris x {n_kolom_bersih} kolom')
size_mb = CLEAN_PATH.stat().st_size / 1e6
print(f'Tersimpan: {CLEAN_PATH} ({size_mb:,.1f} MB)')


Pipeline dieksekusi (streaming sink_parquet) dalam 1.96 menit
Hasil: 27,771,250 baris x 20 kolom
Tersimpan: ../data/processed/idwiki_revisions_clean.parquet (1,791.9 MB)


In [18]:
# Hitung ulang angka sebelum/sesudah langsung dari file (lazy scan - hemat RAM).
# Tidak ada angka hard-coded: seluruh metrik berasal dari data yang dibaca ulang.
m = pl.scan_parquet(DATA_PATH)
b = pl.scan_parquet(CLEAN_PATH)

sebelum = m.select([
    pl.len().alias('total_baris'),
    pl.col('contributor_name').is_null().sum().alias('null_nama'),
    pl.col('rev_id').is_null().sum().alias('null_rev'),
    ((pl.col('text_bytes') < P1) | (pl.col('text_bytes') > P99)).sum().alias('outlier'),
]).collect().row(0, named=True)

sesudah = b.select([
    pl.len().alias('total_baris'),
    pl.col('contributor_name').is_null().sum().alias('null_nama'),
    pl.col('rev_id').is_null().sum().alias('null_rev'),
    ((pl.col('text_bytes_winsorized') < P1) | (pl.col('text_bytes_winsorized') > P99)).sum().alias('outlier'),
    pl.col('is_first_revision').sum().alias('n_first'),
    (~pl.col('has_comment')).sum().alias('n_tanpa_komentar'),
    (~pl.col('is_registered_user')).sum().alias('n_anonim'),
    pl.col('is_outlier_size').sum().alias('n_outlier_flag'),
]).collect().row(0, named=True)

tabel = pl.DataFrame({
    'metrik': [
        'Total baris',
        'Null contributor_name',
        'Null rev_id (kunci)',
        'Baris di luar batas winsor p1..p99',
        'Kolom flag struktural (is_first_revision / has_comment / is_registered_user)',
        'Kolom hasil winsor (text_bytes_winsorized / is_outlier_size)',
    ],
    'sebelum': [
        f"{sebelum['total_baris']:,}",
        f"{sebelum['null_nama']:,}",
        f"{sebelum['null_rev']:,}",
        f"{sebelum['outlier']:,}",
        '-',
        '-',
    ],
    'sesudah': [
        f"{sesudah['total_baris']:,}",
        f"{sesudah['null_nama']:,}",
        f"{sesudah['null_rev']:,}",
        f"{sesudah['outlier']:,}",
        'ditambahkan',
        'ditambahkan',
    ],
})
print('Semua revisi dipertahankan (tidak ada baris yang dihapus).')
print(f"is_first_revision=True  : {sesudah['n_first']:,}")
print(f"has_comment=False       : {sesudah['n_tanpa_komentar']:,}")
print(f"is_registered_user=False: {sesudah['n_anonim']:,}")
print(f"is_outlier_size=True    : {sesudah['n_outlier_flag']:,}")
tabel


Semua revisi dipertahankan (tidak ada baris yang dihapus).
is_first_revision=True  : 4,255,798
has_comment=False       : 5,252,300
is_registered_user=False: 2,199,136
is_outlier_size=True    : 539,133


metrik,sebelum,sesudah
str,str,str
"""Total baris""","""27,771,250""","""27,771,250"""
"""Null contributor_name""","""5,571""","""0"""
"""Null rev_id (kunci)""","""0""","""0"""
"""Baris di luar batas winsor p1.…","""539,133""","""0"""
"""Kolom flag struktural (is_firs…","""-""","""ditambahkan"""
"""Kolom hasil winsor (text_bytes…","""-""","""ditambahkan"""


In [19]:
# Preview 8 baris pertama dari file hasil (lazy, hanya kolom yang dibutuhkan)
pl.scan_parquet(CLEAN_PATH).select([
    'rev_id', 'page_title', 'rev_timestamp', 'contributor_name',
    'is_first_revision', 'has_comment', 'is_registered_user',
    'text_bytes', 'text_bytes_winsorized', 'is_outlier_size',
]).head(8).collect()


rev_id,page_title,rev_timestamp,contributor_name,is_first_revision,has_comment,is_registered_user,text_bytes,text_bytes_winsorized,is_outlier_size
i64,str,datetime[μs],str,bool,bool,bool,i64,i64,bool
1,"""Asam deoksiribonukleat""",2003-11-07 00:43:23,"""proxy8.ntu.edu.sg""",true,true,true,27,27,false
2,"""Asam deoksiribonukleat""",2003-11-07 00:43:35,"""proxy8.ntu.edu.sg""",false,true,true,31,31,false
3,"""Asam deoksiribonukleat""",2003-11-07 00:52:09,"""155.69.5.xxx""",false,true,true,296,296,false
4,"""Asam deoksiribonukleat""",2003-11-07 01:04:46,"""155.69.5.xxx""",false,true,true,639,639,false
5,"""Asam deoksiribonukleat""",2003-11-07 01:43:34,"""155.69.5.xxx""",false,true,true,2281,2281,false
6,"""Asam deoksiribonukleat""",2003-11-07 09:39:07,"""Vyasa""",false,true,true,2871,2871,false
7,"""Asam deoksiribonukleat""",2003-11-07 10:20:52,"""Vyasa""",false,true,true,3163,3163,false
86,"""Asam deoksiribonukleat""",2003-11-07 12:12:58,"""Vyasa""",false,true,true,3161,3161,false


## Catatan Milestone 2

Dokumentasikan di sini: dimensi kualitas data yang paling bermasalah, strategi yang dipilih untuk missing
values dan outlier, jumlah baris yang berubah, serta aturan validasi yang diterapkan.

1. **DuckDB `SUMMARIZE`** menjalankan profil lengkap 15 kolom dalam satu scan — deliverable wajib terpenuhi.
2. **Kualitas kuantitatif**: 0 duplikat `rev_id`, 0 nilai negatif `text_bytes`/`namespace`, timestamp dalam rentang wajar,
   referensial `parent_rev_id` sehat, flag anonim konsisten.
3. **Null struktural ditangani dengan benar**: tanpa imputasi buta; diwakili `is_first_revision`, `has_comment`,
   `is_registered_user`. Hanya `contributor_name` (0,02%) yang diisi label `"unknown"`.
4. **Outlier `text_bytes`**: batas IQR + domain knowledge; winsor p1–p99 (20 – 115.873); baris tidak dihapus.
5. **Validasi dua lapis**: Great Expectations lulus pada sampel ~300 ribu baris; aturan global DuckDB lulus
   pada seluruh 27,7 juta baris.
6. **Hasil akhir**: 27.771.250 baris × 20 kolom disimpan streaming ke `data/processed/idwiki_revisions_clean.parquet`.
